# Example: Analysis of Put Credit Vertical Spreads
In this example, we will analyze vertical bullish credit spreads. These are directional options strategies that involve selling a higher strike put and buying a lower strike put, the high-strike is near the money and the low-strike is out of the money.

> __Learning Objectives:__
> 
> By the end of this example, you will be able to:
> * __Construct profit diagrams for put vertical spreads:__ Build profit and loss profiles for put credit spreads by combining short and long put contracts with different strike prices, and compute the breakeven point where the position transitions from profit to loss.
> * __Calculate probability of profit and probability of exercise for vertical spreads:__ Use cumulative distribution functions of projected share price distributions to determine the likelihood of profit at the breakeven point, and each leg's probability of exercise, under our belief $\mathbb{P}$ and the risk-neutral probabilities $\mathbb{Q}$.
> * __Compute Delta for composite positions to assess hedging requirements:__ Calculate the composite Delta of a put vertical spread by summing the weighted Deltas of its legs on the Leisen–Reimer tree, compare the leg Deltas with the CRR tree and the quoted Deltas, and find the number of shares that makes the position delta-neutral.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
To start, load the options data using [the `MyOptionsChainDataSet(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/data/#VLQuantitativeFinancePackage.MyOptionsChainDataSet) and store the metadata in the `metadata::Dict{String,Any}` variable and the options chain data in the `optionschain::DataFrame` variable.

In [2]:
metadata, optionschain = let
    
    # initialize -
    options_dataset = MyOptionsChainDataSet(ticker="amd-2026");
    metadata = options_dataset.metadata;
    optionschain = options_dataset.data;

    (metadata, optionschain) # return
end;

What's in the `metadata::Dict{String,Any}` dictionary?

In [3]:
metadata |> keys

KeySet for a Dict{String, Any} with 10 entries. Keys:
  "DTE"
  "purchase_date"
  "historical_volatility"
  "underlying_share_price_ask"
  "source"
  "underlying_share_price_bid"
  "is_weekly"
  "atm_IV"
  "expiration_date"
  "underlying_share_price_mid"

What about the `optionschain::DataFrame` variable?

In [4]:
optionschain

Row,Strike,Moneyness,Bid,Mid,Ask,Last,Volume,Open Int,IV,Delta,Type
,Float64,Float64,Float64,Float64,Float64,Float64,Int64,String7,Float64,Float64,String7
1,5.0,0.9906,524.2,526.53,528.85,517.3,0,"1,074",5.2163,0.9996,Call
2,10.0,0.9812,518.9,521.4,523.9,479.51,0,4,3.7202,0.9998,Call
3,15.0,0.9718,514.3,516.63,518.95,0.0,0,0,3.8539,0.9992,Call
4,20.0,0.9624,509.3,511.63,513.95,466.97,0,956,3.4532,0.9992,Call
5,25.0,0.953,504.1,506.55,509.0,427.94,0,212,2.9877,0.9994,Call
6,30.0,0.9436,499.6,501.83,504.05,457.14,0,208,3.167,0.9986,Call
7,35.0,0.9342,494.65,496.85,499.05,479.33,0,9,2.9794,0.9985,Call
8,40.0,0.9247,488.4,491.17,493.95,483.4,1,13,1.9981,0.9999,Call
9,45.0,0.9153,484.25,486.7,489.15,425.63,0,101,2.4434,0.9991,Call


### Constants
Let's set the constants. The share price $S_{0}$ is AMD's close on the quote date. The step size `Δt::Float64` is one day in units of years (a `365-day` year), and `T::Float64` is the time to expiration. The risk-free rate $g_{y}$ is the 13-week Treasury bill yield on the quote date, converted to continuous compounding, and `𝒟::Float64` is the accumulation factor $\mathcal{D}_{T,0}(g_{y})=e^{g_{y}T}$ from today to expiration.

In [5]:
Sₒ = metadata["underlying_share_price_mid"] |> x-> parse(Float64, x); # close on the quote date, 2026-10-26 (USD/share)

Other constants:

In [6]:
Δt = (1/365); # one calendar day, expressed in years
DTE = metadata["DTE"] |> x-> parse(Float64, x); # days to expiration (days)
T = DTE*Δt; # time to expiration (years)
τ_bill = 91/365; # term of the 13-week Treasury bill (years)
y_bill = 0.0415; # 13-week bill coupon-equivalent yield on 2026-10-26, U.S. Treasury Daily Treasury Bill Rates; update it if you change the quote date
# https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_bill_rates&field_tdr_date_value_month=202610
gᵧ = log(1 + y_bill*τ_bill)/τ_bill; # continuously compounded risk-free rate (1/yr)
𝒟 = exp(gᵧ*T); # accumulation factor from today to expiration (dimensionless)

In this example, we'll need to do a projection of the future share price distribution at the contract's expiration date. We'll use a Geometric Brownian Motion (GBM) model to project the future share price distribution at expiration.

Let's load the single asset geometric Brownian motion (SAGBM) parameters which we estimated a few lectures ago, which include the drift and diffusion coefficients for each ticker. We'll store this data in the `parameters_df::DataFrame` variable:

In [7]:
parameters_df = let

    # load -
    df = CSV.read(joinpath(_PATH_TO_DATA,"SAGBM-Parameters-Fall-2025.csv"), DataFrame);
    df; # return
end;

We are looking at `AMD`, so let's see what the parameters for this ticker are:

In [8]:
filter(:ticker=> x-> x=="AMD", parameters_df)

Row,ticker,drift,t,lower_bound_drift,upper_bound_drift,volatility
,String7,Float64,Float64,Float64,Float64,Float64
1,AMD,0.439658,1.96082,0.43459,0.444725,0.474283


### Contract models
Next, we construct contract models for the options in this example. We begin by building an instance of the [MyAmericanPutContractModel type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/derivatives/#VLQuantitativeFinancePackage.MyAmericanPutContractModel) using the [custom build(...) method](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/derivatives/#VLQuantitativeFinancePackage.build-Tuple{Type{MyAmericanPutContractModel},%20NamedTuple}). 

The [build(...) method](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/derivatives/#VLQuantitativeFinancePackage.build-Tuple{Type{MyAmericanPutContractModel},%20NamedTuple}) takes two arguments: the type to build, and a [NamedTuple](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) containing the strike price `K`, the `premium` paid for the option, the days to expiration `DTE` (units: years), the `sense` parameter (1 for long, -1 for short), the number of contracts in the `copy` field and the implied volatility in the `IV` field.

We save the __short__ put option contract model, at the higher strike price $K_{2}$, in the `put_contract_model_short::MyAmericanPutContractModel` variable:

In [9]:
put_contract_model_short = let

    # initialize -
    K = 520.0; # strike price of the short put, K₂ (USD/share)
    options_dataset =  optionschain;
    premium = filter([:Strike,:Type] => (x,y) -> (x == K && y == "Put"), options_dataset)[1,:Mid];
    IV = filter([:Strike,:Type] => (x,y) -> (x == K && y == "Put"), options_dataset)[1,:IV]; # implied volatility (decimal)
    DTE = metadata["DTE"] |> x-> parse(Float64, x); # days to expiration (units: days)
    θ = -1; # short position
    

    # build the contract model -
    put_option_model = build(MyAmericanPutContractModel, (
        K = K, 
        premium = premium, # mark price
        DTE = DTE*Δt, # units: years
        sense = θ,
        copy = 1,
        IV = IV
    ));

    put_option_model; # return the model -
end

MyAmericanPutContractModel(520.0, -1, 0.1452054794520548, 0.5619, 38.1, nothing, 1)

Next, let's build the __long__ put option contract model for the lower strike price $K_{1}$ in the vertical spread. Let's save the long put option contract model in the `put_contract_model_long::MyAmericanPutContractModel` variable:

In [10]:
put_contract_model_long = let

    # initialize -
    K = 500.0; # strike price of the long put, K₁ (USD/share)
    options_dataset =  optionschain;
    premium = filter([:Strike,:Type] => (x,y) -> (x == K && y == "Put"), options_dataset)[1,:Mid];
    IV = filter([:Strike,:Type] => (x,y) -> (x == K && y == "Put"), options_dataset)[1,:IV]; # implied volatility (decimal)
    DTE = metadata["DTE"] |> x-> parse(Float64, x); # days to expiration (units: days)
    θ = 1; # long position

    # build the contract model -
    put_option_model = build(MyAmericanPutContractModel, (
        K = K, 
        premium = premium, # mark price
        DTE = DTE*Δt, # units: years
        sense = θ,
        copy = 1,
        IV = IV
    ));

    put_option_model; # return the model -
end

MyAmericanPutContractModel(500.0, 1, 0.1452054794520548, 0.5669, 29.35, nothing, 1)

## Task 3: Is this trade Delta-neutral?
In this task, we compute the composite Delta of the put credit spread when we open it, and the number of shares that would make the position delta-neutral.

With one copy of each leg, the composite Delta of the spread, per share, is the sum of the legs' Deltas weighted by their orientation:
$$
\Delta_{\mathcal{C}} = \theta_{1}\Delta_{1} + \theta_{2}\Delta_{2} = \Delta_{1} - \Delta_{2}
$$
where $\Delta_{i}$ is the Delta of a long put at strike $K_{i}$. A positive $\Delta_{\mathcal{C}}$ means the position gains when the share price rises. The shares that make one spread delta-neutral are $N_{S} = -m\,\Delta_{\mathcal{C}}$, with $m = 100$ shares per contract.

We estimate each leg's Delta by a central difference on the Leisen–Reimer tree, repricing with the share price moved up and down by $\epsilon_{S}=0.01\,S_{0}$, using [the `𝒫(...)` function](src/Helper.jl) on a tree with $h=20\cdot\text{DTE}+1$ levels (odd, about twenty per day). For comparison, we also compute it on the CRR tree with $h$ and $h+1$ levels, because CRR estimates move back and forth between odd and even numbers of levels, and we show the archive's quoted Delta. Every Delta in the table is the Delta of a long put, before the orientation is applied. So what do we get?

In [21]:
delta_table = let

    # initialize -
    h = 20*round(Int, DTE) + 1; # odd number of tree levels, about twenty per day
    ε_S = 0.01*Sₒ; # share price step (USD/share)
    legs = [("put at K₁ (long leg)", put_contract_model_long), ("put at K₂ (short leg)", put_contract_model_short)];

    # each leg's Delta as a long contract: Leisen–Reimer, CRR with h and h + 1 levels, and the archive's quote -
    df = DataFrame(leg = String[], K = Float64[], Δ_LR = Float64[], Δ_CRR_h = Float64[], Δ_CRR_h1 = Float64[], Δ_quote = Float64[]);
    for (label, model) ∈ legs
        σ = model.IV; # this leg's implied volatility (decimal)
        Δ_LR = (𝒫(model; S = Sₒ + ε_S, σ = σ, T = T, g = gᵧ, h = h) -
            𝒫(model; S = Sₒ - ε_S, σ = σ, T = T, g = gᵧ, h = h))/(2*ε_S);
        Δ_CRR = [(binomial_premium(model; method = :crr, h = n, Sₒ = Sₒ + ε_S, σ = σ, T = T, μ = gᵧ) -
            binomial_premium(model; method = :crr, h = n, Sₒ = Sₒ - ε_S, σ = σ, T = T, μ = gᵧ))/(2*ε_S) for n ∈ (h, h + 1)];
        Δ_quote = filter([:Strike,:Type] => (x,y) -> (x == model.K && y == "Put"), optionschain)[1,:Delta];
        push!(df, (label, model.K, Δ_LR, Δ_CRR[1], Δ_CRR[2], Δ_quote));
    end
    pretty_table(df; column_labels = ["leg (Delta of a long put)", "K (USD/share)", "Leisen–Reimer", "CRR, h = $(h)", "CRR, h = $(h + 1)", "archive quote"],
        alignment = [:l, :r, :r, :r, :r, :r], formatters = [fmt__printf("%.4f", [3, 4, 5, 6])],
        table_format = TextTableFormat(borders = text_table_borders__compact),
        fit_table_in_display_horizontally = false);

    # checks: a long put's Delta is between -1 and 0 -
    @assert all(-1 .≤ df.Δ_LR .≤ 0) "a put Delta is outside [-1, 0]"

    df; # return
end;

 --------------------------- --------------- --------------- --------------- --------------- ---------------
  leg (Delta of a long put)   K (USD/share)   Leisen–Reimer   CRR, h = 1061   CRR, h = 1062   archive quote 
 --------------------------- --------------- --------------- --------------- --------------- ---------------
  put at K₁ (long leg)                500.0         -0.3405         -0.3410         -0.3401         -0.3406
  put at K₂ (short leg)               520.0         -0.4097         -0.4107         -0.4089         -0.4099
 --------------------------- --------------- --------------- --------------- --------------- ---------------


Now let's compute the composite Delta of the spread and the shares that make it delta-neutral, using the Leisen–Reimer Deltas:

In [22]:
let

    # initialize: Δ_𝒞 = θ₁Δ₁ + θ₂Δ₂ with θ₁ = 1 (long put at K₁) and θ₂ = -1 (short put at K₂) -
    m = 100; # shares per contract
    Δ₁ = delta_table.Δ_LR[1]; # Delta of a long put at K₁
    Δ₂ = delta_table.Δ_LR[2]; # Delta of a long put at K₂ (we are short this leg)
    Δ_𝒞 = Δ₁ - Δ₂; # composite Delta of the spread (per share)
    N_S = -m*Δ_𝒞; # shares that make one spread delta-neutral

    # show the results -
    println("Composite Delta of the spread (per share), Δ_𝒞 = Δ₁ - Δ₂ = $(round(Δ_𝒞, digits = 4))");
    println("Shares that make one spread delta-neutral, N_S = -100⋅Δ_𝒞 = $(round(N_S, digits = 2))");
    println(N_S < 0 ? "The spread gains when the share price rises, so we would sell (short) $(round(-N_S, digits = 1)) shares to hedge it." :
        "The spread gains when the share price falls, so we would buy $(round(N_S, digits = 1)) shares to hedge it.");
end

Composite Delta of the spread (per share), Δ_𝒞 = Δ₁ - Δ₂ = 0.0692
Shares that make one spread delta-neutral, N_S = -100⋅Δ_𝒞 = -6.92
The spread gains when the share price rises, so we would sell (short) 6.9 shares to hedge it.


> __What do we see?__
>
> * __Leisen–Reimer and the quotes:__ The Leisen–Reimer Deltas match the archive's quoted Deltas to about three decimal places. The CRR Deltas with $h$ and $h+1$ levels sit on either side of them.
> * __The spread is nearly delta-neutral:__ The two legs' Deltas nearly cancel, so the spread's composite Delta is small and positive. It gains a little when the share price rises, as a bullish position should, and hedging one spread takes selling only a few shares.

___